In [ ]:
!pip install -q "siftfs @ git+https://github.com/yafi38/sift-fs@01b677d1aaffc08b50332d3eca77cd72d58edfbb"

# sift-fs grid search

Hyperparameter search for the `sift-fs` feature-selection gating layer, run on Kaggle GPU notebooks against the MNIST and Activity (AAL) datasets.

**Scope.** This notebook installs `siftfs` pinned to commit `01b677d` of `main` and then reimplements the training loop directly against internal pieces of the library (`_MaskedMLP`, `FeatureSelectorLoss`) instead of the public `FeatureSelector` API. That's intentional: it lets us try training-loop variations (reheat, annealing, alternate hidden-dim configs) directly in the notebook without needing a matching code change in the `siftfs` repo first. The tradeoff is that this training loop can drift from what `FeatureSelector.fit()` actually does upstream — if the public API's training/selection logic changes, this notebook won't pick it up automatically.

**Data.** Expects preprocessed `.npy` arrays uploaded as a Kaggle dataset input under `/kaggle/input/datasets/mashrurahmedyafi/{activity-search,mnist-search}/`. See `benchmarks/data_prep/` in the repo for how those were produced from the raw Activity/AAL and MNIST sources.

**Reading the results.** Each grid search writes one timestamped run directory per hyperparameter combination (`history.json`, `config.json`, loss/weight-convergence plots) plus a single `accuracy_results.csv` summarizing all combinations in that sweep. `classifier_accuracy` is the downstream signal that matters most: it's the test accuracy of an ExtraTrees classifier trained only on the panel of features the gating layer selected. `top_k_jaccard_last_10` measures how stable the selected feature set is over the final 10 epochs (1.0 = fully converged, not still swapping features in/out).

In [ ]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path
from typing import Literal
import time

import matplotlib
import numpy as np
import torch
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch import nn

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

from siftfs import FeatureSelectorLoss  # noqa: E402
from siftfs.selector import _MaskedMLP  # noqa: E402


In [ ]:
DATA_DIR = Path("/kaggle/input/datasets/mashrurahmedyafi")
OUT_DIR = Path.cwd()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def load_dataset(dataset: Literal["activity", "mnist"]) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if dataset == "activity":
        activity_dir = DATA_DIR / "activity-search"
        X = np.load(activity_dir / "activity_X_search_train.npy").astype(np.float32)
        y = np.load(activity_dir / "activity_y_search_train.npy")

        X_test = np.load(activity_dir / "activity_X_search_eval.npy").astype(np.float32)
        y_test = np.load(activity_dir / "activity_y_search_eval.npy")
    elif dataset == "mnist":
        mnist_dir = DATA_DIR / "mnist-search"
        X = np.load(mnist_dir / "mnist_X_search_train.npy").astype(np.float32)
        y = np.load(mnist_dir / "mnist_y_search_train.npy")

        X_test = np.load(mnist_dir / "mnist_X_search_eval.npy").astype(np.float32)
        y_test = np.load(mnist_dir / "mnist_y_search_eval.npy")
    else:
        raise ValueError(f"Unknown dataset {dataset}")

    return X, y, X_test, y_test

In [ ]:
REHEAT_SETTLE_EPOCHS = 20


def scheduled_l1(
    epoch: int,
    epochs: int,
    l1: float,
    reheat: tuple[int, int, float] | None,
    anneal: tuple[float, int] | None,
) -> float:
    if anneal is not None:
        start_l1, anneal_till = anneal
        frac = min(epoch / anneal_till, 1.0)
        return start_l1 * (l1 / start_l1) ** frac
    if reheat is not None:
        every, window, factor = reheat
        settling = epochs - epoch <= REHEAT_SETTLE_EPOCHS
        in_burst = not settling and epoch % every < window
        return l1 * factor if in_burst else l1
    return l1


def train(
    X: np.ndarray,
    y: np.ndarray,
    *,
    panel_size: int = 50,
    epochs: int = 200,
    alpha: float = 1.5,
    beta: float = 0.2,
    gamma: float = 0.5,
    l1: float = 0.01,
    l2_decay: float = 0.01,
    hidden_dims: tuple[int, ...] = (32, 16),
    init: float = 0.5,
    lr: float = 1e-3,
    batch_size: int = 64,
    validation_split: float = 0.2,
    seed: int = 33,
    device: str = "cpu",
    reheat: tuple[int, int, float] | None = None,
    anneal: tuple[float, int] | None = None,
) -> tuple[list[dict[str, object]], np.ndarray, dict[str, object]]:
    labels = np.unique(y)
    enc = np.searchsorted(labels, y)
    n_cls, n_feat = len(labels), X.shape[1]
    
    X_tr, X_va, y_tr, y_va = train_test_split(
        X,
        enc,
        test_size=validation_split,
        random_state=seed,
        stratify=enc,
    )
    class_weights = compute_class_weight("balanced", classes=np.arange(n_cls), y=y_tr)
    cw = torch.tensor(class_weights, dtype=torch.float32, device=device)
    
    torch.manual_seed(seed)
    if device.startswith("cuda"):
        torch.cuda.manual_seed_all(seed)
    generator = torch.Generator().manual_seed(seed)
    
    model = _MaskedMLP(n_feat, n_cls, hidden_dims, init).to(device)
    loss_fn = FeatureSelectorLoss(
        model.gating,
        panel_size=panel_size,
        alpha=alpha,
        beta=beta,
        gamma=gamma,
        strict=True,
        l1=l1,
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Move the full (small) dataset to the device once. Per-batch host<->device
    # transfers and per-batch `.item()` syncs otherwise dominate wall time on GPU
    # for a model this size, since there's not enough compute per batch to hide
    # the transfer/launch/sync overhead.
    X_tr_t = torch.from_numpy(X_tr).to(device)
    y_tr_t = torch.from_numpy(y_tr).to(device)
    X_va_t = torch.from_numpy(X_va).to(device)
    y_va_t = torch.from_numpy(y_va).to(device)
    
    history: list[dict[str, object]] = []
    prev_top: set[int] | None = None
    n_train = len(X_tr)
    n_val = len(X_va)
    
    for epoch in range(epochs):
        loss_fn.l1 = scheduled_l1(epoch, epochs, l1, reheat, anneal)
    
        model.train()
        tot_train = torch.zeros((), device=device)
        task_train = torch.zeros((), device=device)
        perm = torch.randperm(n_train, generator=generator).to(device)
        for start in range(0, n_train, batch_size):
            idx = perm[start : start + batch_size]
            xb = X_tr_t[idx]
            yb = y_tr_t[idx]
            optimizer.zero_grad()
            logits = model(xb)
            task_loss = nn.functional.cross_entropy(logits, yb, weight=cw)
            total = loss_fn(task_loss) + 0.5 * l2_decay * model.l2_penalty()
            total.backward()
            optimizer.step()
            tot_train += total.detach() * len(xb)
            task_train += task_loss.detach() * len(xb)
        tot_train = tot_train.item()
        task_train = task_train.item()
    
        model.eval()
        val_loss_t = torch.zeros((), device=device)
        val_correct = 0
        with torch.no_grad():
            for start in range(0, n_val, batch_size):
                xb = X_va_t[start : start + batch_size]
                yb = y_va_t[start : start + batch_size]
                logits = model(xb)
                val_loss_t += nn.functional.cross_entropy(logits, yb, weight=cw) * len(xb)
                val_correct += int((logits.argmax(1) == yb).sum())
        val_loss = val_loss_t.item()
    
        w = model.gating.weight.detach().cpu().numpy()
        aw = np.abs(w)
        top = set(np.argsort(aw)[::-1][:panel_size])
        jaccard = len(top & prev_top) / panel_size if prev_top is not None else 1.0
        prev_top = top
    
        binary_force = float(np.sum(aw * np.abs(w - 1)))
        count_term = alpha * abs(float(aw.sum()) - panel_size)
        fs_loss = loss_fn.l1 * (binary_force + count_term)
    
        history.append(
            {
                "epoch": epoch,
                "l1": loss_fn.l1,
                "train_total": tot_train / n_train,
                "train_task": task_train / n_train,
                "reg": (tot_train - task_train) / n_train,
                "val_loss": val_loss / n_val,
                "val_acc": val_correct / n_val,
                "sum_abs": float(aw.sum()),
                "mean_abs": float(aw.mean()),
                "binary_force": binary_force,
                "count_term": count_term,
                "fs_loss": fs_loss,
                "n_over_0.9": int(np.sum(aw > 0.9)),
                "n_under_0.1": int(np.sum(aw < 0.1)),
                "top_k_jaccard": jaccard,
            }
        )

    config = {
        "panel_size": panel_size,
        "epochs": epochs,
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma,
        "l1": l1,
        "l2_decay": l2_decay,
        "hidden_dims": hidden_dims,
        "init": init,
        "lr": lr,
        "batch_size": batch_size,
        "validation_split": validation_split,
        "seed": seed,
        "device": device,
        "reheat": reheat,
        "anneal": anneal,
    }

    return history, model.gating.weight.detach().cpu().numpy(), config

In [ ]:
def plot_weights(
    history: list[dict[str, object]],
    panel_size: int,
    out: Path,
) -> None:
    ep = [int(h["epoch"]) for h in history]

    fig = plt.figure(figsize=(15, 4))
    gs = fig.add_gridspec(1, 2, width_ratios=[2.4, 1])

    ax = fig.add_subplot(gs[0])
    ax.plot(ep, [h["top_k_jaccard"] for h in history], color="tab:purple")
    ax.set_xlabel("epoch")
    ax.set_ylabel("top-k set overlap (Jaccard)")
    ax.set_title("Selection set convergence")
    ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[1])
    ax.plot(ep, [h["sum_abs"] for h in history], label="sum |w|", color="tab:blue")
    ax.axhline(panel_size, color="grey", linewidth=0.8, linestyle="--", label=f"d={panel_size}")
    ax.set_xlabel("epoch")
    ax.set_ylabel("sum |w|")
    ax.set_title("Count constraint")
    ax.legend()
    ax.grid(alpha=0.3)

    ax2 = ax.twinx()
    ax2.plot(ep, [h["binary_force"] for h in history], label="binary force", color="tab:orange")
    ax2.set_ylabel("binary force", color="tab:orange")
    ax2.tick_params(axis="y", labelcolor="tab:orange")

    fig.tight_layout()
    fig.savefig(out, dpi=140)

    plt.close()
    print(f"saved {out}")

In [ ]:
def plot_curves(history: list[dict[str, object]], out: Path) -> None:
    ep = [int(h["epoch"]) for h in history]
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))

    axes[0, 0].plot(ep, [h["train_total"] for h in history], label="train total")
    axes[0, 0].plot(ep, [h["train_task"] for h in history], label="train task")
    axes[0, 0].plot(ep, [h["reg"] for h in history], label="reg (FS + L2)")
    axes[0, 0].set_xlabel("epoch")
    axes[0, 0].set_ylabel("loss")
    axes[0, 0].set_title("Loss")
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(ep, [h["fs_loss"] for h in history], color="tab:blue")
    axes[0, 1].set_xlabel("epoch")
    axes[0, 1].set_ylabel("FS loss")
    axes[0, 1].set_title("Feature-selection loss")
    axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(ep, [h["val_loss"] for h in history], color="tab:red")
    axes[1, 0].set_xlabel("epoch")
    axes[1, 0].set_ylabel("val loss")
    axes[1, 0].set_title("Validation loss")
    axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(ep, [h["val_acc"] for h in history], color="tab:green")
    axes[1, 1].set_xlabel("epoch")
    axes[1, 1].set_ylabel("val accuracy")
    axes[1, 1].set_title("Validation accuracy")
    axes[1, 1].grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(out, dpi=140)

    plt.close()
    print(f"saved {out}")


In [ ]:
def get_accuracy_on_classifier(
    X: np.ndarray,
    y: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    weights: np.ndarray,
    panel_size: int,
    seed: int
) -> float:
    selected = np.argsort(np.abs(weights))[::-1][: panel_size]
    clf = ExtraTreesClassifier(n_estimators=50, random_state=seed)
    clf.fit(X[:, selected], y)
    test_acc = clf.score(X_test[:, selected], y_test)

    return test_acc

In [ ]:
def run_test(
    dataset: Literal["activity", "mnist"],
    X: np.ndarray,
    y: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    alpha: float,
    l1: float,
    epochs: int,
    seed: int,
    panel_size: int,
    full_outdir: Path,
    reheat: tuple[int, int, float] | None = None,
    anneal: tuple[float, int] | None = None,
    hidden_dims: tuple[int, ...] = (32, 16),
    save_stats: bool = True,
) -> dict:
    print(f"\nRunning with alpha = {alpha}, l1 = {l1}, epochs = {epochs}, seed = {seed}, reheat = {reheat}, anneal = {anneal}, and hidden_dims = {hidden_dims}.")

    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")

    history, weights, config = train(
        X,
        y,
        alpha=alpha,
        l1=l1,
        panel_size=panel_size,
        epochs=epochs,
        seed=seed,
        device=DEVICE,
        reheat=reheat,
        anneal=anneal,
        hidden_dims=hidden_dims
    )

    if save_stats:
        run_outdir = full_outdir / stamp
        run_outdir.mkdir(parents=True, exist_ok=True)
    
        hist_path = run_outdir / "history.json"
        hist_path.write_text(json.dumps(history))
    
        config_path = run_outdir / "config.json"
        config_path.write_text(json.dumps(config))

        plot_curves(history, run_outdir / "curves.png")
        plot_weights(history, panel_size, run_outdir / "weights.png")

    acc = get_accuracy_on_classifier(
        X,
        y,
        X_test,
        y_test,
        weights,
        panel_size,
        seed=seed
    )

    print(f"Classifier accuracy (extra trees): {acc:.4f}")

    last = history[-1]
    val_acc = last['val_acc']
    val_loss = last['val_loss']
    sum_w = last['sum_abs']

    last10_jaccard = np.mean([h["top_k_jaccard"] for h in history[-10:]])

    return {
        "id": stamp,
        "dataset": dataset,
        "alpha": alpha,
        "l1": l1,
        "epochs": epochs,
        "seed": seed,
        "classifier_accuracy": acc,
        "validation_accuracy": val_acc,
        "validation_loss": val_loss,
        "sum_w": sum_w,
        "top_k_jaccard_last_10": last10_jaccard
    }

## Sweep: `alpha` / `l1`

Coarse sweep over the count-penalty strength (`alpha`) and overall regularization strength (`l1`), the two knobs that most directly control how aggressively the gating layer prunes down to `panel_size` features. No reheat/anneal schedule — `l1` is held constant across training.

In [ ]:
def run_grid_search_on_alpha_l1(dataset: Literal["activity", "mnist"]) -> None:
    alphas = [0.25, 0.50, 0.75, 1.0, 1.25, 1.50]
    l1s = [0.001, 0.003, 0.01, 0.03, 0.1]
    seeds = [33, 42]
    epochs_list = [200]
    PANEL_SIZE = 50

    X, y, X_test, y_test = load_dataset(dataset)

    full_outdir = OUT_DIR / "grid_alpha_l1" / datetime.now().strftime("%Y%m%d-%H%M%S")
    result_rows: list[dict] = []

    for alpha in alphas:
        for l1 in l1s:
            for epochs in epochs_list:
                for seed in seeds:
                    start_time = time.perf_counter()
                    
                    result_row = run_test(
                        dataset, X, y, X_test, y_test, alpha, l1, epochs, seed, PANEL_SIZE, full_outdir
                    )
                    result_rows.append(result_row)
                    
                    result_path = full_outdir / "accuracy_results.csv"
                    pd.DataFrame(result_rows).to_csv(result_path, index=False)
        
                    end_time = time.perf_counter()
                    execution_time = end_time - start_time
                    
                    print(f"Execution time: {execution_time:.6f} seconds")

    print("\n\nSuccesfully ran grid search on alpha and l1.")
    print(f"Results are in {full_outdir}")

## Sweep: reheat schedule

Periodically drops `l1` to `l1 * reheat_factor` for a `reheat_window`-epoch burst every `reheat_every` epochs (disabled for the last 20 epochs so the final selection has time to settle back to the base `l1`). The idea is to let the gating weights escape a locally-converged feature set and re-explore, rather than freezing early on a suboptimal panel.

In [ ]:
def run_grid_search_on_reheat(dataset: Literal["activity", "mnist"]) -> None:
    alphas = [1.00, 1.25]
    l1s = [0.003, 0.01]
    seeds = [33, 42]
    epochs_list = [300]
    PANEL_SIZE = 50
    reheat_everys = [30, 40, 50]
    reheat_windows = [5, 10, 15, 20]
    reheat_factors = [0.1, 0.01, 0.001]

    X, y, X_test, y_test = load_dataset(dataset)

    full_outdir = OUT_DIR / "grid_reheat" / datetime.now().strftime("%Y%m%d-%H%M%S")
    result_rows: list[dict] = []

    for alpha in alphas:
        for l1 in l1s:
            for epochs in epochs_list:
                for reheat_every in reheat_everys:
                    for reheat_window in reheat_windows:
                        for reheat_factor in reheat_factors:
                            for seed in seeds:
                                start_time = time.perf_counter()

                                reheat = reheat_every, reheat_window, reheat_factor
                                
                                result_row = run_test(
                                    dataset, X, y, X_test, y_test, alpha, l1, epochs, seed, PANEL_SIZE, full_outdir, reheat
                                )

                                result_row.update({
                                    "reheat_every": reheat_every,
                                    "reheat_window": reheat_window,
                                    "reheat_factor": reheat_factor,
                                })
                                result_rows.append(result_row)
                                
                                result_path = full_outdir / "accuracy_results.csv"
                                pd.DataFrame(result_rows).to_csv(result_path, index=False)
                    
                                end_time = time.perf_counter()
                                execution_time = end_time - start_time
                                
                                print(f"Execution time: {execution_time:.6f} seconds")

    print("\n\nSuccesfully ran grid search on reheat.")
    print(f"Results are in {full_outdir}")

## Sweep: `l1` annealing

Instead of a fixed `l1`, ramps it geometrically from `start_l1` up to the target `l1` over `anneal_till` epochs (`l1_t = start_l1 * (l1 / start_l1) ** min(epoch / anneal_till, 1)`), so the gating layer starts nearly unregularized and the count/selection pressure builds in gradually.

This grid was narrowed over three successive passes as earlier results ruled out ranges:

- **Run 1** (Activity): `alpha in [1.00, 1.25]`, `l1 in [0.01, 0.03]`, `epochs=300`, `start_l1 in [1e-3, 1e-4, 1e-5]`, `anneal_till in [150, 200, 250]`.
- **Run 2** (Activity): narrowed to `alpha in [1.25, 1.5]`, `epochs=450`, `start_l1 in [1e-4, 1e-5]`, `anneal_till in [300, 350]`, added `seed=7`.
- **Run 3** (MNIST, active below): same `start_l1`/`anneal_till` range as Run 2, extended `alpha` to `[1.0, 1.25, 1.5]`, `epochs=400`, to check whether the Activity-tuned range transfers to MNIST.

In [ ]:
def run_grid_search_on_anneal(dataset: Literal["activity", "mnist"]) -> None:
    # Run 3 -- MNIST (see markdown above for Run 1/Run 2 history)
    alphas = [1.0, 1.25, 1.5]
    l1s = [0.01, 0.03]
    seeds = [33, 42, 7]
    epochs_list = [400]
    PANEL_SIZE = 50
    start_l1s = [0.0001, 0.00001]
    anneal_tills = [300, 350]

    X, y, X_test, y_test = load_dataset(dataset)

    full_outdir = OUT_DIR / "grid_anneal" / datetime.now().strftime("%Y%m%d-%H%M%S")
    result_rows: list[dict] = []

    for alpha in alphas:
        for l1 in l1s:
            for epochs in epochs_list:
                for start_l1 in start_l1s:
                    for anneal_till in anneal_tills:
                        for seed in seeds:
                            start_time = time.perf_counter()

                            anneal = start_l1, anneal_till
                            
                            result_row = run_test(
                                dataset, X, y, X_test, y_test, alpha, l1, epochs, seed, PANEL_SIZE, full_outdir, anneal=anneal
                            )

                            result_row.update({
                                "start_l1": start_l1,
                                "anneal_till": anneal_till,
                            })
                            result_rows.append(result_row)
                            
                            result_path = full_outdir / "accuracy_results.csv"
                            pd.DataFrame(result_rows).to_csv(result_path, index=False)
                
                            end_time = time.perf_counter()
                            execution_time = end_time - start_time
                            
                            print(f"Execution time: {execution_time:.6f} seconds")

    print("\n\nSuccesfully ran grid search on anneal.")
    print(f"Results are in {full_outdir}")

## Sweep: hidden dims

Holds `alpha`, `l1`, and the annealing schedule fixed at the best-performing values found above, and instead varies the task MLP's hidden-layer widths to check whether task-network capacity changes which features get selected or how well they generalize.

In [ ]:
def run_grid_search_on_hidden_dims(dataset: Literal["activity", "mnist"]) -> None:
    alpha = 1.25
    l1 = 0.03
    # seeds = [33, 42, 7]
    seeds = [79, 93]
    epochs = 400
    PANEL_SIZE = 50
    start_l1 = 0.00001
    anneal_till = 300
    hidden_dims_list = [(16, 8), (32, 16), (128, 64), (256, 128), (128, 64, 32)]

    X, y, X_test, y_test = load_dataset(dataset)
    
    full_outdir = OUT_DIR / "grid_hidden_dims" / datetime.now().strftime("%Y%m%d-%H%M%S")
    result_rows: list[dict] = []

    for hidden_dims in hidden_dims_list:
        for seed in seeds:
            start_time = time.perf_counter()
    
            anneal = start_l1, anneal_till
            
            result_row = run_test(
                dataset, X, y, X_test, y_test, alpha, l1, epochs, seed, PANEL_SIZE, full_outdir, anneal=anneal, hidden_dims=hidden_dims
            )
    
            result_row.update({
                "start_l1": start_l1,
                "anneal_till": anneal_till,
                "hidden_dims": hidden_dims
            })
            result_rows.append(result_row)
            
            result_path = full_outdir / "accuracy_results.csv"
            pd.DataFrame(result_rows).to_csv(result_path, index=False)
    
            end_time = time.perf_counter()
            execution_time = end_time - start_time
            
            print(f"Execution time: {execution_time:.6f} seconds")

    print("\n\nSuccesfully ran grid search on hidden dims.")
    print(f"Results are in {full_outdir}")

In [ ]:
run_grid_search_on_anneal("mnist")